In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder.appName("DT_CV").getOrCreate()

26/04/14 00:17:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Load dataset from GCS
data = spark.read.csv("gs://spark_mllib/train_data", header=True, inferSchema=True)
data.show()

+-----------------+----------------+-----------------+----------------+-----+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|label|
+-----------------+----------------+-----------------+----------------+-----+
|              5.1|             3.5|              1.4|             0.2|    0|
|              4.9|             3.0|              1.4|             0.2|    0|
|              4.7|             3.2|              1.3|             0.2|    0|
|              4.6|             3.1|              1.5|             0.2|    0|
|              5.0|             3.6|              1.4|             0.2|    0|
|              5.4|             3.9|              1.7|             0.4|    0|
|              4.6|             3.4|              1.4|             0.3|    0|
|              5.0|             3.4|              1.5|             0.2|    0|
|              4.4|             2.9|              1.4|             0.2|    0|
|              4.9|             3.1|              1.5|          

In [3]:
# Clean column names
for c in data.columns:
    data = data.withColumnRenamed(
        c, c.replace(" ", "_").replace("(", "").replace(")", "")
    )

label_col = "label"
feature_cols = [c for c in data.columns if c != label_col]

print(feature_cols)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

dt = DecisionTreeClassifier(
    labelCol=label_col,
    featuresCol="features"
)

pipeline = Pipeline(stages=[assembler, dt])

# Train
model = pipeline.fit(data)

# Predict
predictions = model.transform(data)
predictions.select("features", "label", "prediction").show()

['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm']


+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[5.1,3.5,1.4,0.2]|    0|       0.0|
|[4.9,3.0,1.4,0.2]|    0|       0.0|
|[4.7,3.2,1.3,0.2]|    0|       0.0|
|[4.6,3.1,1.5,0.2]|    0|       0.0|
|[5.0,3.6,1.4,0.2]|    0|       0.0|
|[5.4,3.9,1.7,0.4]|    0|       0.0|
|[4.6,3.4,1.4,0.3]|    0|       0.0|
|[5.0,3.4,1.5,0.2]|    0|       0.0|
|[4.4,2.9,1.4,0.2]|    0|       0.0|
|[4.9,3.1,1.5,0.1]|    0|       0.0|
|[5.4,3.7,1.5,0.2]|    0|       0.0|
|[4.8,3.4,1.6,0.2]|    0|       0.0|
|[4.8,3.0,1.4,0.1]|    0|       0.0|
|[4.3,3.0,1.1,0.1]|    0|       0.0|
|[5.8,4.0,1.2,0.2]|    0|       0.0|
|[5.7,4.4,1.5,0.4]|    0|       0.0|
|[5.4,3.9,1.3,0.4]|    0|       0.0|
|[5.1,3.5,1.4,0.3]|    0|       0.0|
|[5.7,3.8,1.7,0.3]|    0|       0.0|
|[5.1,3.8,1.5,0.3]|    0|       0.0|
+-----------------+-----+----------+
only showing top 20 rows



In [5]:
# Hyperparameter grid
paramGrid = (ParamGridBuilder()
             .addGrid(dt.maxDepth, [3, 5, 10])
             .addGrid(dt.minInstancesPerNode, [1, 5, 10])
             .build())

evaluator = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
)

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

In [6]:
# Train
cv_model = crossval.fit(data)

In [7]:
# Best model
best_model = cv_model.bestModel

In [8]:
# Extract best parameters
best_dt = best_model.stages[-1]

print("Best maxDepth:", best_dt.getMaxDepth())
print("Best minInstancesPerNode:", best_dt.getMinInstancesPerNode())

Best maxDepth: 3
Best minInstancesPerNode: 1


In [9]:
# Save model to GCS
model_path = "gs://spark_mllib/models/dt_model"
best_model.write().overwrite().save(model_path)

In [10]:
model = pipeline.fit(data)

model.write().overwrite().save("gs://spark_mllib/models/final_dt_model")

In [11]:
spark.stop()